In [1]:
#importuje napisane przez nas funkcje i je aktualizuje na bieżąco, dzięki czemu nie trzeba restartować kernela przy każdej zmianie w funkcjach
%load_ext autoreload
%autoreload 2

from funkcje import *

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv("zbior_1.csv")
X=df.iloc[:, :-1]
y=df.iloc[:, -1]

#Podzil na zbior treningowy (70%) i pozostale
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=2137,
    stratify=y
)

#Podzil na zbior walidacyjny (15%) i testowy (15%)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=2137,
    stratify=y_temp
)

In [4]:
#sprawdzenie czy istnieja powtarzajace sie wiersze w df
df.duplicated().sum()

np.int64(0)

In [5]:
#wypisanie nazw kolumn ktore maja typ int lub str
print(f"kolumny z typem int: {X_train.select_dtypes(include=['int']).columns}")
print(f"kolumny z typem str: {X_train.select_dtypes(include=['str']).columns}")

kolumny z typem int: Index(['szczegolnaFormaPrawna_Symbol', 'formaWlasnosci_Symbol', 'pkdKod',
       'wsk_liczba_dni_istnienia'],
      dtype='str')
kolumny z typem str: Index(['schemat_wsk_bilans', 'schemat_wsk_rzis'], dtype='str')


In [6]:
#podzial na kolumny numeryczne i kategoryczne
numeric_columns = X_train.select_dtypes(include=['float']).columns
numeric_columns = numeric_columns.append(pd.Index(['wsk_liczba_dni_istnienia']))
categorical_columns = X_train.select_dtypes(include=['str', 'int']).columns
categorical_columns = categorical_columns.drop('wsk_liczba_dni_istnienia')

X_cat=X_train[categorical_columns]
X_num=X_train[numeric_columns]

In [7]:
#zstopienie wartości nieskończonych w kolumnach numerycznych przez nan
X_num = X_num.replace([np.inf, -np.inf], np.nan)

In [8]:
#wypisanie kolumn numerycznych z brakami danych
print("Braki danych w kolumnach numerycznych:")
for c in X_num.columns:
    if X_num[c].isna().any():
        print(f"{c}: {X_num[c].isna().sum()/len(X_num):.2%} braków danych")

Braki danych w kolumnach numerycznych:
wsk_kapital_do_aktywa: 1.62% braków danych
RP_przeplywy_operacyjne: 99.19% braków danych
RP_amortyzacja: 95.48% braków danych
wsk_akt_generowania_got_2: 99.19% braków danych
wsk_pokrycie_wyd_fin_gotowkowe_2: 99.19% braków danych
wsk_pokrycie_zob_kr_gotowkowe_2: 99.19% braków danych
wsk_zadluzenia_gotowki_2: 99.19% braków danych
wsk_plynnosc_biez_1: 35.00% braków danych
wsk_zysk_CF_operacyjny: 99.19% braków danych
wsk_struktura_kap_wlasnego_s_2: 3.62% braków danych


In [9]:
#wypianie kolumn kategorycznych z brakami danych
print("Braki danych w kolumnach kategorycznych:")
for c in X_cat.columns:
    if X_cat[c].isna().any():
        print(f"{c}: {X_cat[c].isna().sum()/len(X_cat):.2%} braków danych")

Braki danych w kolumnach kategorycznych:


In [10]:
#wypisanie jaki procent danych w kolumnach numerycznych to 0 malejąco
procent_zer = (
    X_num.eq(0)
    .agg(["sum", "mean"])
    .T
    .rename(columns={
        "sum": "liczba_zer",
        "mean": "część_zer"
    })
    .query("część_zer > 0")
    .sort_values("część_zer", ascending=False)
)

procent_zer.index.name = "nazwa_kolumny"
with pd.option_context("display.max_rows", None):
    display(procent_zer)

,liczba_zer,część_zer
nazwa_kolumny,,
Zobowiazania_dostaw_uslug_pow12m_kapitale,2100.0,1.000000
Naleznosci_dostaw_uslug_pow12m_kapitale,2098.0,0.999048
Ujemna_wartosc_firmy,2097.0,0.998571
Naleznosci_dostaw_uslug_pow12m_powiazane,2097.0,0.998571
Zobowiazania_dostaw_uslug_pow12m_powiazane,2095.0,0.997619
Zobowiazania_dostaw_uslug_pow12m,2090.0,0.995238
Pozostale_rezerwy,2087.0,0.993810
Wartosc_firmy,2086.0,0.993333
Zobowiazania_dostaw_uslug_pow12m_pozostale,2084.0,0.992381


In [11]:
#policzenie jednowymiarowego AUC dla każdej kolumny numerycznej, Małysz twierdził, że to pomaga zobaczyć jak cech jest ważna
cechy_num = [c for c in X_num.columns]
rows = []
for c in cechy_num:

    m = X_num[c].notna()
    auc = roc_auc_score(y_train.loc[m], X_num.loc[m, c])
    rows.append({"cecha": c, "AUC jednowymiarowe": max(auc, 1 - auc), "kierunek": "+" if auc >= 0.5 else "-", "braki": 1 - m.mean()})
tab_auc = pd.DataFrame(rows).set_index("cecha").sort_values("AUC jednowymiarowe", ascending=False)
with pd.option_context("display.max_rows", None):
    display(tab_auc.round(3))
podejrzane = tab_auc.index[tab_auc["AUC jednowymiarowe"] > 0.85].tolist()
print("Podejrzanie silne cechy (AUC > 0,85):", podejrzane)

,AUC jednowymiarowe,kierunek,braki
cecha,,,
wsk_zysk_CF_operacyjny,0.800,-,0.992
wsk_index_skr_zaw,0.700,+,0.000
wsk_index_nowe,0.694,+,0.000
wsk_pokrycie_zob_kr_gotowkowe_2,0.667,+,0.992
wsk_ebitda_2,0.666,-,0.000
wsk_ebitda_1,0.665,-,0.000
zysk_brutto,0.665,-,0.000
Kapital_wlasny,0.664,-,0.000
wsk_pokrycie_wyd_fin_gotowkowe_1,0.664,-,0.000


Podejrzanie silne cechy (AUC > 0,85): []


In [12]:
#policzenie korelacji pomiędzy kolumnami numerycznymi i wypisanie tych z korelacją większą niż 0,9999
korelacje = X_num.corr()

# Pobranie tylko górnego trójkąta macierzy, bez przekątnej
maska = np.triu(np.ones(korelacje.shape), k=1).astype(bool)

pary_korelacji = (
    korelacje.where(maska)
    .stack()
    .reset_index()
)

pary_korelacji.columns = ["cecha_1", "cecha_2", "korelacja"]

# Najsilniejsze zależności: dodatnie i ujemne
pary_korelacji["abs_korelacja"] = pary_korelacji["korelacja"].abs()

pary_korelacji = (
    pary_korelacji
    .sort_values("abs_korelacja", ascending=False)
    .drop(columns="abs_korelacja")
    .reset_index(drop=True)
)

#wypisanie par cech z korelacją większą niż 0,85
pary_korelacji.query("korelacja > 0.9999 or korelacja < -0.9999")


,cecha_1,cecha_2,korelacja
0,wsk_ebitda_koszty_odsetkowe_2,wsk_ebitda_zobowiazan_odsetki_2,1.000000
1,wsk_ebitda_koszty_odsetkowe_1,wsk_ebitda_zobowiazan_odsetki_1,1.000000
2,wsk_koszty_odsetki_przychody,wsk_zadluzenie_gotowka,1.000000
3,wsk_zysk_op_zobowiazania,wsk_zysk_operacyjny_zob_1,1.000000
4,wsk_kapital_do_aktywa,wsk_struktura_kap_wlasnego_1,1.000000
5,wsk_koszty_przychody,wsk_poziom_kosztow_operacyjnych,1.000000
6,wsk_kapital_do_aktywa,wsk_udzial_kap_wlasnego_aktywa_1,1.000000
7,wsk_udzial_kap_wlasnego_aktywa_1,wsk_struktura_kap_wlasnego_1,1.000000
8,wsk_koszty_fin_przychody,wsk_poziom_kosztow_finansowych,1.000000
9,wsk_ROS,wsk_rent_sprzedazy,1.000000


In [31]:
#testowanie funkcji usuwania kolumn o wysokiej korelacji
df_cleaned = usuwanie_najbardziej_scorrelowanych(X_num, prog=0.999999)
print(f"Usunięto następujące kolumny: {set(X_num.columns) - set(df_cleaned.columns)}")

Usunięto 15 kolumn.
Usunięto następujące kolumny: {'wsk_stopa_zysku_sprzedaz', 'wsk_kapital_do_aktywa', 'wsk_zadluzenie_kap_wlasnego', 'wsk_udzial_kap_wlasnego_aktywa_1', 'wsk_ebitda_koszty_odsetkowe_1', 'wsk_zysk_op_zobowiazania', 'wsk_ROS', 'wsk_ebitda_koszty_odsetkowe_2', 'wsk_koszty_fin_przychody', 'wsk_udzial_zob_biez_sprzedaz_1', 'wsk_koszty_przychody', 'Zobowiazania_krotkoterminowe', 'wsk_koszty_odsetki_przychody', 'wsk_pokrycie_aktywow_kap_stalym', 'wsk_ROE'}


In [13]:
#Wyszukanie outliers w kolumnach numerycznych za pomocą metody IQR i wypisanie ich liczby w całym zbiorze treningowym
outliers = {}
for column in X_num.columns:
    Q1 = X_num[column].quantile(0.25)
    Q3 = X_num[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 3 * IQR
    upper_bound = Q3 + 3* IQR
    outliers[column] = X_num[(X_num[column] < lower_bound) | (X_num[column] > upper_bound)][column]


outliers_count = {column: len(outliers[column]) for column in outliers}

outliers_count_df = pd.DataFrame(
    list(outliers_count.items()),
    columns=['column', 'outliers_count']
)

# Liczba unikalnych wierszy zawierających przynajmniej jeden outlier
outlier_rows = set()

for column, values in outliers.items():
    outlier_rows.update(values.index)

print("Liczba wierszy oznaczonych jako outlier:", len(outlier_rows))
with pd.option_context("display.max_rows", None):
    display(outliers_count_df.sort_values(by='outliers_count', ascending=False))


Liczba wierszy oznaczonych jako outlier: 2099


,column,outliers_count
88,wsk_stopa_marzy_brutto,576
89,wsk_rent_operacyjna,574
130,wsk_zadluzenia_gotowki_1,549
150,wsk_pokrycia_odsetek_zyskiem,547
71,przychody_odsetki,520
45,Kredyty_pozyczki_krotkie,493
115,wsk_zysk_op_zobowiazania,472
128,wsk_zysk_operacyjny_zob_1,472
187,wsk_udzial_zob_biez_sprzedaz_2,467
101,wsk_rent_sprzedazy,458
